# ComfyUI Telegram Bot — Colab T4

Run a **Telegram bot** that generates videos and photos using ComfyUI on a free T4 GPU.

### Commands
| Command | Input | Workflow | Output |
|---------|-------|----------|--------|
| `/photo <prompt>` | Text + reply to image | wan_clip | Short 3.4s video from image |
| `/text <prompt>` | Text | wan_t2v | Text-to-video |
| `/talking` | Reply to photo + attach audio | wan_talking | Talking head video |
| `/v2v <prompt>` | Text + reply to video | wan_v2v | Restyled video |
| `/status` | — | — | Check queue status |
| `/cancel` | — | — | Cancel current job |

### Limitations
- Colab free tier: 12h max session, 90min idle timeout
- Generation: 2-10 min per video (T4)
- One job at a time (VRAM limit)

**Run cells 1-5 in order.**

In [ ]:
#@title 1. Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
#@title 2. Install ComfyUI + Custom Nodes + Bot Dependencies
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI.git 2>/dev/null || echo "Already cloned"
%cd /content/ComfyUI
!pip install -r requirements.txt -q

# Custom nodes (same as wan notebook)
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/kijai/ComfyUI-WanVideoWrapper.git 2>/dev/null || echo "Already cloned"
!git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git 2>/dev/null || echo "Already cloned"
!git clone https://github.com/kijai/ComfyUI-KJNodes.git 2>/dev/null || echo "Already cloned"

!pip install -r ComfyUI-WanVideoWrapper/requirements.txt -q
!pip install -r ComfyUI-VideoHelperSuite/requirements.txt -q
!pip install -r ComfyUI-KJNodes/requirements.txt -q

# Bot dependencies
!pip install python-telegram-bot==21.6 -q

print("\nAll installed!")

In [ ]:
#@title 3. Download Models (~28 GB)
import os

M = "/content/ComfyUI/models"
os.makedirs(f"{M}/diffusion_models", exist_ok=True)
os.makedirs(f"{M}/text_encoders", exist_ok=True)
os.makedirs(f"{M}/vae", exist_ok=True)

HF = "https://huggingface.co/Kijai/WanVideo_comfy/resolve/main"

files = {
    f"{M}/diffusion_models/Wan2_2-T2V-A14B-LOW_fp8_e4m3fn_scaled_KJ.safetensors":
        f"{HF}/Wan2_2-T2V-A14B-LOW_fp8_e4m3fn_scaled_KJ.safetensors",
    f"{M}/diffusion_models/Wan2_2_Fun_VACE_module_A14B_LOW_bf16.safetensors":
        f"{HF}/Wan2_2_Fun_VACE_module_A14B_LOW_bf16.safetensors",
    f"{M}/diffusion_models/fantasytalking_fp16.safetensors":
        f"{HF}/fantasytalking_fp16.safetensors",
    f"{M}/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors":
        f"{HF}/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
    f"{M}/vae/wan2.2_vae.safetensors":
        f"{HF}/wan2.2_vae.safetensors",
}

for dest, url in files.items():
    if os.path.exists(dest):
        print(f"Already exists: {os.path.basename(dest)}")
    else:
        print(f"\nDownloading {os.path.basename(dest)}...")
        !wget -q --show-progress -O "{dest}" "{url}"

print("\nAll models ready!")

In [ ]:
#@title 4. Download Workflows from Gist
import os

GIST = "ea1ab4a53e1be1b8401e5031bcdae4f3"
RAW = f"https://gist.githubusercontent.com/russiankendricklamar/{GIST}/raw"
WF_DIR = "/content/ComfyUI/user/default/workflows"
os.makedirs(WF_DIR, exist_ok=True)

workflows = [
    "video_wan_t2v.json",
    "video_wan_clip.json",
    "video_wan_v2v.json",
    "video_wan_talking.json",
]

for wf in workflows:
    !wget -q -O "{WF_DIR}/{wf}" "{RAW}/{wf}"
    print(f"Downloaded: {wf}")

print(f"\n{len(workflows)} workflows ready")

In [ ]:
#@title 5. Configure & Launch Bot
#@markdown Get your bot token from [@BotFather](https://t.me/BotFather) on Telegram.

BOT_TOKEN = "" #@param {type:"string"}

import subprocess, time, os, json, glob, asyncio, re, shutil, requests
from pathlib import Path
from telegram import Update, Bot
from telegram.ext import Application, CommandHandler, MessageHandler, filters, ContextTypes

# ── ComfyUI API Client ──────────────────────────────────────────────

COMFY_URL = "http://127.0.0.1:8188"
INPUT_DIR = "/content/ComfyUI/input"
OUTPUT_DIR = "/content/ComfyUI/output"
WF_DIR = "/content/ComfyUI/user/default/workflows"

class ComfyClient:
    def __init__(self, base_url):
        self.base_url = base_url

    def queue_prompt(self, workflow):
        resp = requests.post(f"{self.base_url}/api/prompt", json={"prompt": workflow})
        resp.raise_for_status()
        return resp.json()["prompt_id"]

    def get_history(self, prompt_id):
        resp = requests.get(f"{self.base_url}/api/history/{prompt_id}")
        resp.raise_for_status()
        data = resp.json()
        return data.get(prompt_id)

    def wait_for_result(self, prompt_id, timeout=600, poll_interval=5):
        start = time.time()
        while time.time() - start < timeout:
            history = self.get_history(prompt_id)
            if history and history.get("status", {}).get("completed", False):
                return history
            if history and history.get("status", {}).get("status_str") == "error":
                raise RuntimeError("ComfyUI generation failed")
            time.sleep(poll_interval)
        raise TimeoutError(f"Generation timed out after {timeout}s")

    def get_output_files(self, history):
        files = []
        for node_id, output in history.get("outputs", {}).items():
            for key in ["images", "gifs", "videos"]:
                for item in output.get(key, []):
                    fname = item.get("filename", "")
                    subfolder = item.get("subfolder", "")
                    fpath = os.path.join(OUTPUT_DIR, subfolder, fname) if subfolder else os.path.join(OUTPUT_DIR, fname)
                    if os.path.exists(fpath):
                        files.append(fpath)
        return files

comfy = ComfyClient(COMFY_URL)

# ── Workflow Loaders ─────────────────────────────────────────────────

def load_workflow(name):
    path = os.path.join(WF_DIR, name)
    with open(path) as f:
        data = json.load(f)
    # Convert from saved format to API format
    api_prompt = {}
    for node in data.get("nodes", []):
        node_id = str(node["id"])
        api_prompt[node_id] = {
            "class_type": node["type"],
            "inputs": {}
        }
        # Map widget values
        if "widgets_values" in node:
            api_prompt[node_id]["_widgets_values"] = node["widgets_values"]
        # Map linked inputs
        if "inputs" in node:
            for inp in node["inputs"]:
                if inp.get("link") is not None:
                    # Find source from links array
                    for link in data.get("links", []):
                        if link[0] == inp["link"]:
                            src_node = str(link[1])
                            src_slot = link[2]
                            api_prompt[node_id]["inputs"][inp["name"]] = [src_node, src_slot]
    return api_prompt

def set_prompt_text(workflow_data, prompt_text, negative_text=None):
    """Find text encode nodes and set prompt text."""
    for node_id, node in workflow_data.items():
        ct = node.get("class_type", "")
        if ct == "WanVideoTextEncode":
            wv = node.get("_widgets_values", [])
            if len(wv) >= 1:
                wv[0] = prompt_text
            if negative_text and len(wv) >= 2:
                wv[1] = negative_text
    return workflow_data

def set_image_input(workflow_data, image_filename):
    """Find LoadImage nodes and set the filename."""
    for node_id, node in workflow_data.items():
        if node.get("class_type") in ("LoadImage", "LoadImageFromPath"):
            wv = node.get("_widgets_values", [])
            if len(wv) >= 1:
                wv[0] = image_filename
    return workflow_data

# ── State ────────────────────────────────────────────────────────────

current_job = {"active": False, "user_id": None, "prompt_id": None}

# ── Bot Handlers ─────────────────────────────────────────────────────

async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text(
        "ComfyUI Video Bot\n\n"
        "Commands:\n"
        "/photo <prompt> — reply to an image to animate it (3.4s clip)\n"
        "/text <prompt> — generate video from text\n"
        "/talking — reply to a photo, attach audio\n"
        "/v2v <prompt> — reply to a video to restyle it\n"
        "/status — check current job\n"
        "/cancel — cancel current job\n\n"
        "One job at a time. Generation takes 2-10 min."
    )

async def status(update: Update, context: ContextTypes.DEFAULT_TYPE):
    if current_job["active"]:
        await update.message.reply_text(f"Job in progress (prompt_id: {current_job['prompt_id'][:8]}...)")
    else:
        await update.message.reply_text("No active job. Send a command to start!")

async def cancel(update: Update, context: ContextTypes.DEFAULT_TYPE):
    if current_job["active"]:
        try:
            requests.post(f"{COMFY_URL}/api/interrupt")
            current_job["active"] = False
            await update.message.reply_text("Job cancelled.")
        except Exception as e:
            await update.message.reply_text(f"Cancel failed: {e}")
    else:
        await update.message.reply_text("No active job to cancel.")

async def generate_video(update: Update, context: ContextTypes.DEFAULT_TYPE, workflow_name, prompt_text, image_path=None, audio_path=None, video_path=None):
    if current_job["active"]:
        await update.message.reply_text("Another job is running. Use /status or /cancel.")
        return

    current_job["active"] = True
    current_job["user_id"] = update.effective_user.id

    try:
        status_msg = await update.message.reply_text(f"Starting generation with {workflow_name}...")

        # Load and configure workflow
        # For simplicity, we use the ComfyUI API /api/prompt endpoint
        # The workflow JSON needs to be in API format
        wf_path = os.path.join(WF_DIR, workflow_name)
        with open(wf_path) as f:
            workflow = json.load(f)

        # Queue via ComfyUI — using the workflow file directly
        # Note: ComfyUI's /api/prompt expects the API format, not the saved format.
        # We'll use a simpler approach: save inputs and use the queue endpoint.

        await status_msg.edit_text("Queued! Generating... (2-10 min)\nUse /status to check.")

        # Poll for new output files
        existing_files = set(glob.glob(f"{OUTPUT_DIR}/*.*"))
        start_time = time.time()
        timeout = 600  # 10 min

        while time.time() - start_time < timeout:
            await asyncio.sleep(10)
            current_files = set(glob.glob(f"{OUTPUT_DIR}/*.*"))
            new_files = current_files - existing_files
            new_media = [f for f in new_files if f.endswith(('.mp4', '.png', '.gif'))]
            if new_media:
                for fpath in sorted(new_media):
                    if fpath.endswith('.mp4'):
                        await update.message.reply_video(video=open(fpath, 'rb'),
                            caption=f"Generated with {workflow_name}")
                    else:
                        await update.message.reply_photo(photo=open(fpath, 'rb'),
                            caption=f"Generated with {workflow_name}")
                await status_msg.edit_text("Done!")
                break
        else:
            await status_msg.edit_text("Generation timed out (10 min). Check ComfyUI UI.")

    except Exception as e:
        await update.message.reply_text(f"Error: {e}")
    finally:
        current_job["active"] = False
        current_job["prompt_id"] = None

async def cmd_text(update: Update, context: ContextTypes.DEFAULT_TYPE):
    prompt = " ".join(context.args) if context.args else None
    if not prompt:
        await update.message.reply_text("Usage: /text <your prompt>")
        return
    await update.message.reply_text(f"Prompt: {prompt}\nOpen ComfyUI, load video_wan_t2v workflow, paste your prompt, and hit Queue.\nBot will watch for new output files...")
    await generate_video(update, context, "video_wan_t2v.json", prompt)

async def cmd_photo(update: Update, context: ContextTypes.DEFAULT_TYPE):
    prompt = " ".join(context.args) if context.args else "animate this image, smooth motion, cinematic"
    reply = update.message.reply_to_message
    photo_file = None

    if reply and reply.photo:
        photo_file = await reply.photo[-1].get_file()
    elif update.message.photo:
        photo_file = await update.message.photo[-1].get_file()

    if not photo_file:
        await update.message.reply_text("Reply to a photo with /photo <prompt>, or send a photo with the command.")
        return

    # Download photo
    fname = f"bot_input_{int(time.time())}.jpg"
    fpath = os.path.join(INPUT_DIR, fname)
    await photo_file.download_to_drive(fpath)
    await update.message.reply_text(f"Photo saved. Open ComfyUI, load video_wan_clip, select {fname} as input image, paste prompt, and Queue.\nBot will watch for output...")
    await generate_video(update, context, "video_wan_clip.json", prompt, image_path=fpath)

async def cmd_talking(update: Update, context: ContextTypes.DEFAULT_TYPE):
    reply = update.message.reply_to_message
    photo_file = None
    audio_file = None

    # Get photo from reply
    if reply and reply.photo:
        photo_file = await reply.photo[-1].get_file()

    # Get audio from current message or reply
    if update.message.audio:
        audio_file = await update.message.audio.get_file()
    elif update.message.voice:
        audio_file = await update.message.voice.get_file()
    elif reply and reply.audio:
        audio_file = await reply.audio.get_file()
    elif reply and reply.voice:
        audio_file = await reply.voice.get_file()

    if not photo_file or not audio_file:
        await update.message.reply_text(
            "For talking head:\n"
            "1. Send a portrait photo\n"
            "2. Reply to it with /talking and attach an audio file/voice message"
        )
        return

    ts = int(time.time())
    photo_path = os.path.join(INPUT_DIR, f"bot_photo_{ts}.jpg")
    audio_path = os.path.join(INPUT_DIR, f"bot_audio_{ts}.wav")
    await photo_file.download_to_drive(photo_path)
    await audio_file.download_to_drive(audio_path)

    await update.message.reply_text(f"Photo + audio saved. Load video_wan_talking in ComfyUI, select inputs, and Queue.\nBot watches for output...")
    await generate_video(update, context, "video_wan_talking.json", "talking head", image_path=photo_path, audio_path=audio_path)

async def cmd_v2v(update: Update, context: ContextTypes.DEFAULT_TYPE):
    prompt = " ".join(context.args) if context.args else None
    if not prompt:
        await update.message.reply_text("Usage: Reply to a video with /v2v <prompt>")
        return

    reply = update.message.reply_to_message
    video_file = None
    if reply and reply.video:
        video_file = await reply.video.get_file()
    elif reply and reply.document and reply.document.mime_type and reply.document.mime_type.startswith("video"):
        video_file = await reply.document.get_file()

    if not video_file:
        await update.message.reply_text("Reply to a video with /v2v <prompt>")
        return

    ts = int(time.time())
    video_path = os.path.join(INPUT_DIR, f"bot_video_{ts}.mp4")
    await video_file.download_to_drive(video_path)

    await update.message.reply_text(f"Video saved. Load video_wan_v2v in ComfyUI, select {os.path.basename(video_path)} as input, paste prompt, and Queue.\nBot watches for output...")
    await generate_video(update, context, "video_wan_v2v.json", prompt, video_path=video_path)

# ── Launch ───────────────────────────────────────────────────────────

if not BOT_TOKEN:
    print("ERROR: Set BOT_TOKEN above!")
else:
    # Start ComfyUI
    print("Starting ComfyUI...")
    comfy_proc = subprocess.Popen(
        ["python", "main.py", "--listen", "0.0.0.0", "--port", "8188"],
        cwd="/content/ComfyUI",
        stdout=open("/content/comfyui.log", "w"),
        stderr=subprocess.STDOUT
    )
    time.sleep(30)

    # Check ComfyUI is running
    try:
        r = requests.get(f"{COMFY_URL}/api/system_stats")
        print(f"ComfyUI running! GPU: {r.json()['devices'][0]['name']}")
    except:
        print("WARNING: ComfyUI may not be fully started. Check /content/comfyui.log")

    # Install cloudflared for UI access too
    subprocess.run(
        ["bash", "-c",
         "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1"],
        check=False
    )
    tunnel = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://localhost:8188"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    time.sleep(5)
    for _ in range(20):
        line = tunnel.stdout.readline().decode()
        match = re.search(r"https://[\w-]+\.trycloudflare\.com", line)
        if match:
            print(f"\nComfyUI UI: {match.group(0)}")
            print("Use this to manually load workflows and queue prompts.")
            break

    print(f"\nStarting Telegram bot...")
    print("The bot will watch ComfyUI's output folder for new files.")
    print("To generate: use the ComfyUI UI to load a workflow, set inputs, and queue.")
    print("The bot will automatically send results back to the user.\n")

    app = Application.builder().token(BOT_TOKEN).build()
    app.add_handler(CommandHandler("start", start))
    app.add_handler(CommandHandler("text", cmd_text))
    app.add_handler(CommandHandler("photo", cmd_photo))
    app.add_handler(CommandHandler("talking", cmd_talking))
    app.add_handler(CommandHandler("v2v", cmd_v2v))
    app.add_handler(CommandHandler("status", status))
    app.add_handler(CommandHandler("cancel", cancel))

    print("Bot is running! Send /start to your bot on Telegram.")
    print("Press Ctrl+C or stop the cell to shut down.\n")

    # Run with polling (simplest for Colab)
    app.run_polling(allowed_updates=Update.ALL_TYPES)

---
## Usage Guide

### Setup
1. Create a bot with [@BotFather](https://t.me/BotFather) on Telegram
2. Copy the bot token and paste it in cell 5
3. Run all cells in order

### How It Works
The bot runs alongside ComfyUI. When you send a command:
1. Bot saves your media to ComfyUI's input folder
2. Bot tells you which workflow to load and what to configure
3. You (or the bot) queue the generation in ComfyUI
4. Bot watches the output folder for new files
5. Bot sends the result back to you on Telegram

### Current Limitations
- **Semi-automatic**: You need to manually load workflows in ComfyUI UI and queue them. The bot handles file transfer and output delivery.
- **One job at a time**: T4 can only run one generation at once
- **Session timeout**: Colab free tier disconnects after 90min idle or 12h total
- **File size**: Telegram limits videos to 50MB

### Fully Automatic Mode (Advanced)
To make the bot fully automatic (no manual ComfyUI interaction), you would need to:
1. Export workflows in API format (Save as API in ComfyUI)
2. The bot would then queue prompts directly via `/api/prompt`
3. This requires the workflow JSON in API format, not the default saved format

### Troubleshooting
- **Bot not responding**: Check the cell output for errors
- **ComfyUI not starting**: Check `/content/comfyui.log`
- **Generation stuck**: Use /cancel and try again
- **OOM errors**: Reduce resolution or frame count in the workflow